
<br>
task10_feasibility_demo.py<br>
================================================================================<br>
Milestone-1 Task 10 / Demonstration Cases: "Mission test: complete one<br>
feasible payload mission and one deliberately infeasible mission, explaining<br>
the first violated constraint."<br>
Uses mission_planner.py (Task 9) unchanged -- Task 10's feasibility checks<br>
(fuel, power margin, stall, tip Mach, collective/RPM bounds) are already<br>
built into MissionPlanner._step(); this script just demonstrates them on two<br>
concrete missions and prints/saves a clear report of what happened, when,<br>
and why.<br>
================================================================================<br>


In [1]:
from __future__ import annotations
import os
import csv
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Mission planner and tiltrotor design imports

In [2]:
from mission_planner import MissionPlanner
from task5_tiltrotor_design import AIRCRAFT

In [3]:
warnings.filterwarnings("ignore")

In [4]:
FIG_DIR = "figures"
OUT_DIR = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

In [5]:
def _get_fail_attr(fail_obj, attr: str, default=None):
    """Safely extracts attributes from a failure object or dictionary."""
    if fail_obj is None:
        return default
    if isinstance(fail_obj, dict):
        return fail_obj.get(attr, default)
    return getattr(fail_obj, attr, default)

================================================================================<br>
Mission A: FEASIBLE round-trip payload mission<br>
  sized using the validated numbers from Tasks 6/7/9:<br>
    - takeoff weight 2700 kg (below the 3205 kg max-hover-GW-at-SL found<br>
      in Task 6, leaving climb/cruise power margin)<br>
    - climb/descent at 1.5 m/s (found sustainable in Task 9 dev testing)<br>
    - cruise at 22 m/s (inside the ~27.5 m/s trim-feasible envelope found<br>
      in Task 9's cruise-range study)<br>
================================================================================

In [6]:
def feasible_mission():
    mp = MissionPlanner(gross_weight_kg=2700.0, fuel_kg=AIRCRAFT["fuel_capacity_kg"], dt_s=30.0)
    segments = [
        {"type": "hover", "duration_s": 60, "altitude_m": 0.0},
        {"type": "climb", "target_altitude_m": 300.0, "climb_rate_ms": 1.5},
        {"type": "cruise", "distance_km": 30.0, "altitude_m": 300.0, "airspeed_ms": 22.0},
        {"type": "payload", "delta_kg": -150.0},     # drop 150 kg payload
        {"type": "loiter", "duration_s": 120, "altitude_m": 300.0},
        {"type": "cruise", "distance_km": 30.0, "altitude_m": 300.0, "airspeed_ms": 22.0},
        {"type": "descent", "target_altitude_m": 0.0, "descent_rate_ms": 1.5},
        {"type": "hover", "duration_s": 30, "altitude_m": 0.0},
    ]
    summary = mp.run(segments)
    print("=== Task 10, Mission A (FEASIBLE) -- design test case ===")
    print(f"Segments: {[s['type'] for s in segments]}")
    print(f"Completed: {summary['completed']}")
    print(f"Total time: {summary['final_time_s']/60:.1f} min, "
          f"total distance: {summary['final_distance_km']:.1f} km")
    print(f"Fuel remaining: {summary['final_fuel_kg']:.1f} kg "
          f"(reserve = {mp.reserve_fuel_kg:.1f} kg, margin = "
          f"{summary['final_fuel_kg']-mp.reserve_fuel_kg:.1f} kg)")
    print(f"Final gross weight: {summary['final_gross_weight_kg']:.1f} kg")
    _plot_mission(mp, "Mission A (feasible)", "task10_missionA_feasible")
    _write_log_csv(mp, "task10_missionA_feasible_log.csv")
    return mp, summary

================================================================================<br>
Mission B: DELIBERATELY INFEASIBLE mission<br>
  Same starting point as Mission A (2700 kg, a feasible takeoff weight),<br>
  but this time the aircraft picks up a payload mid-mission that is far<br>
  too heavy (+900 kg -- pushing gross weight to 3520 kg, above both the<br>
  3000 kg design MTOW and the 3205 kg max hover weight found in Task 6).<br>
  This is a more illustrative failure than an overweight takeoff: the<br>
  early hover/climb/cruise segments succeed normally, and the mission<br>
  only becomes infeasible once the oversized pickup happens -- exactly<br>
  the kind of segment/time/reason identification Task 10 asks for.<br>
================================================================================

In [7]:
def infeasible_mission():
    mp = MissionPlanner(gross_weight_kg=2700.0, fuel_kg=AIRCRAFT["fuel_capacity_kg"], dt_s=30.0)
    segments = [
        {"type": "hover", "duration_s": 60, "altitude_m": 0.0},
        {"type": "climb", "target_altitude_m": 300.0, "climb_rate_ms": 1.5},
        {"type": "cruise", "distance_km": 30.0, "altitude_m": 300.0, "airspeed_ms": 22.0},
        {"type": "descent", "target_altitude_m": 0.0, "descent_rate_ms": 1.5},
        {"type": "hover", "duration_s": 30, "altitude_m": 0.0},
        {"type": "payload", "delta_kg": +900.0},     # oversized pickup -> infeasible from here on
        {"type": "climb", "target_altitude_m": 300.0, "climb_rate_ms": 1.5},
        {"type": "cruise", "distance_km": 30.0, "altitude_m": 300.0, "airspeed_ms": 22.0},
        {"type": "descent", "target_altitude_m": 0.0, "descent_rate_ms": 1.5},
        {"type": "hover", "duration_s": 30, "altitude_m": 0.0},
    ]
    summary = mp.run(segments)
    print("\n=== Task 10, Mission B (DELIBERATELY INFEASIBLE) -- design test case ===")
    print(f"Takeoff weight: 2700 kg (feasible, same as Mission A) -- but this mission")
    print(f"picks up an oversized +900 kg payload mid-route (-> 3520 kg gross, above the")
    print(f"3000 kg design MTOW and the 3205 kg Task-6 max-hover-GW-at-SL)")
    print(f"Completed: {summary['completed']}")
    failure_info = summary.get("failure") or getattr(mp, "failure", None)
    if failure_info:
        seg_name = _get_fail_attr(failure_info, "segment") or _get_fail_attr(failure_info, "segment_label", "unknown")
        fail_time = _get_fail_attr(failure_info, "time_s", 0.0)
        reason_str = _get_fail_attr(failure_info, "reason", "unknown constraint violation")
        print(f"FIRST VIOLATED CONSTRAINT:")
        print(f"  Segment : {seg_name}")
        print(f"  Time    : {fail_time:.1f} s ({fail_time/60:.1f} min)")
        print(f"  Reason  : {reason_str}")
    else:
        print("(Unexpectedly completed -- adjust the overweight test case.)")
    _plot_mission(mp, "Mission B (deliberately infeasible)", "task10_missionB_infeasible")
    _write_log_csv(mp, "task10_missionB_infeasible_log.csv")
    return mp, summary

================================================================================<br>
Plot & log helpers<br>
================================================================================

In [8]:
def _plot_mission(mp, title: str, fname_prefix: str):
    t_min = [row["t_s"] / 60.0 for row in mp.log if "gross_weight_kg" in row and "t_s" in row]
    gw = [row["gross_weight_kg"] for row in mp.log if "gross_weight_kg" in row]
    alt = [row.get("altitude_m", 0.0) for row in mp.log if "gross_weight_kg" in row]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].plot(t_min, gw, "-o", ms=3, color="tab:blue")
    axes[0].set_xlabel("Time [min]")
    axes[0].set_ylabel("Gross weight [kg]")
    axes[0].set_title("Gross weight vs time")
    axes[0].grid(alpha=0.3)
    axes[1].plot(t_min, alt, "-o", ms=3, color="tab:orange")
    axes[1].set_xlabel("Time [min]")
    axes[1].set_ylabel("Altitude [m]")
    axes[1].set_title("Altitude profile")
    axes[1].grid(alpha=0.3)
    if mp.failed:
        fail_time = _get_fail_attr(mp.failure, "time_s", 0.0) / 60.0
        axes[0].axvline(fail_time, color="red", ls="--", lw=1.5, label="mission failure")
        axes[1].axvline(fail_time, color="red", ls="--", lw=1.5, label="mission failure")
        axes[0].legend()
        axes[1].legend()
    fig.suptitle(f"Task 10 -- {title}")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"{fname_prefix}.png"), dpi=160)
    plt.close(fig)

In [9]:
def _write_log_csv(mp, fname: str):
    if not mp.log:
        return
    keys = set()
    for row in mp.log:
        keys.update(row.keys())
    sorted_keys = sorted(keys)
    filepath = os.path.join(OUT_DIR, fname)
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=sorted_keys)
        writer.writeheader()
        for row in mp.log:
            writer.writerow({k: row.get(k, "") for k in sorted_keys})

In [10]:
if __name__ == "__main__":
    mpA, summaryA = feasible_mission()
    mpB, summaryB = infeasible_mission()
    with open(os.path.join(OUT_DIR, "task10_summary.md"), "w") as f:
        f.write("# Task 10 -- Mission feasibility demonstration summary\n\n")
        f.write("## Mission A (feasible)\n")
        f.write(f"- Completed: {summaryA['completed']}\n")
        f.write(f"- Total time: {summaryA['final_time_s']/60:.1f} min\n")
        f.write(f"- Total distance: {summaryA['final_distance_km']:.1f} km\n")
        f.write(f"- Fuel remaining: {summaryA['final_fuel_kg']:.1f} kg "
                f"(reserve={mpA.reserve_fuel_kg:.1f} kg)\n\n")
        f.write("## Mission B (deliberately infeasible)\n")
        f.write(f"- Completed: {summaryB['completed']}\n")
        failure_info = summaryB.get("failure") or getattr(mpB, "failure", None)
        if failure_info:
            seg_name = _get_fail_attr(failure_info, "segment") or _get_fail_attr(failure_info, "segment_label", "unknown")
            fail_time = _get_fail_attr(failure_info, "time_s", 0.0)
            reason_str = _get_fail_attr(failure_info, "reason", "unknown constraint violation")
            f.write(f"- First violated constraint: segment='{seg_name}', "
                    f"time={fail_time:.1f} s, reason: {reason_str}\n")
    print(f"\nPlots written to {FIG_DIR}/task10_mission[AB]_*.png")
    print(f"Logs written to {OUT_DIR}/task10_mission[AB]_*_log.csv")
    print(f"Summary written to {OUT_DIR}/task10_summary.md")

=== Task 10, Mission A (FEASIBLE) -- design test case ===
Segments: ['hover', 'climb', 'cruise', 'payload', 'loiter', 'cruise', 'descent', 'hover']
Completed: True
Total time: 55.6 min, total distance: 60.0 km
Fuel remaining: 444.6 kg (reserve = 50.0 kg, margin = 394.6 kg)
Final gross weight: 2494.6 kg

=== Task 10, Mission B (DELIBERATELY INFEASIBLE) -- design test case ===
Takeoff weight: 2700 kg (feasible, same as Mission A) -- but this mission
picks up an oversized +900 kg payload mid-route (-> 3520 kg gross, above the
3000 kg design MTOW and the 3205 kg Task-6 max-hover-GW-at-SL)
Completed: False
FIRST VIOLATED CONSTRAINT:
  Segment : climb
  Time    : 1853.6 s (30.9 min)
  Reason  : power required 925.2 kW exceeds power available 757.5 kW

Plots written to figures/task10_mission[AB]_*.png
Logs written to outputs/task10_mission[AB]_*_log.csv
Summary written to outputs/task10_summary.md
